# Method convergence: hypotheses, refinement and asymptotic evidence

Each method tutorial first writes and solves its local and global equations. This companion lesson reads the attributed numerical records and recomputes their convergence diagnostics. Reading records does not rerun the PDE acquisitions.

Check the approximation spaces, physical observable, refinement variable and source SHA256 before interpreting a slope. A spatial energy theorem, a temporal integration order and a difference to a numerical time reference establish different properties. Keep the coarse levels visible and examine several consecutive terminal intervals.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/3c816ab2f78b7b7f37eca70598637a87ca780d21f1ecd64e9e2aed30c4230af4/method_rates-companion.zip"
COMPANION_SHA256 = "3c816ab2f78b7b7f37eca70598637a87ca780d21f1ecd64e9e2aed30c4230af4"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("convergence/method_rates.ipynb", directory=ROOT)


In [ ]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("convergence/method_rates.ipynb")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from examples.tutorial_convergence import asymptotic_summary, method_series, plot_method_series

studies = method_series(ROOT)
print("Recorded studies:", [study["method"] for study in studies])

## 1. Identify the refinement variable and the cited physical norm

For consecutive measured errors, compute the order using the actual refinement ratio, including non-dyadic mesh sizes:

$$
r_i=\frac{\log(E_{i-1}/E_i)}{\log(h_{i-1}/h_i)}.
$$

The reader requires positive finite errors and at least four decreasing sizes. It uses measured PyMHM norms, never digitized points from a figure. Its metadata distinguish literature targets from observations without a stated theorem. A numerical reference must itself be refined before its differences are used as errors.


In [ ]:
method = "unfitted"  # Change this to any name printed above.
study = next(item for item in studies if item["method"] == method)
for key in ("record", "sha256", "refinement", "spaces", "rate_provenance"):
    print(f"{key}: {study[key]}")
for field, errors in study["errors"].items():
    print(field, "errors:", errors)
    print(field, "successive orders:", study["rates"][field])

## 2. Check errors, orders and error amplitudes together

The upper panel retains every measured level and overlays target slopes anchored to the finest error. The lower left panel shows every successive order. The lower right divides each error by the corresponding target power:

$$
A_i=\frac{E_i}{h_i^q},\qquad
\widetilde A_i=\frac{A_i}{A_{\mathrm{finest}}}.
$$

In an asymptotic regime, the orders approach the literature target and the normalized amplitudes approach a horizontal plateau. The shaded region always contains the last four measured levels, giving three consecutive rates. It does not select convenient intervals. An error floor, unstable rate or increasing amplitude remains visible. Sparse tick labels preserve every plotted sample and remain readable at documentation width.


In [ ]:
figure = plot_method_series(study)
plt.show()
output = ROOT / "build" / "method-convergence"
output.mkdir(parents=True, exist_ok=True)
credit = "IPES Research Group; CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/"
for extension in ("png", "svg", "pdf"):
    metadata = {"Author": credit} if extension == "pdf" else (
        {"Creator": credit} if extension == "svg" else {"Author": credit}
    )
    figure.savefig(output / f"{method}-convergence.{extension}", dpi=160, bbox_inches="tight", metadata=metadata)


## 3. Inspect a common terminal window for every observable

Report three consecutive rates, a log–log fitted order on the same four levels and the variation of the target-normalized error amplitude. These descriptive diagnostics do not replace local-discretization checks, independent quadrature or verification of the theorem's assumptions. A good fitted slope alone can conceal an unsuitable terminal interval.


In [ ]:
for study in studies:
    print("\n", study["method"], "—", study["refinement"])
    for field, diagnostic in asymptotic_summary(study).items():
        print(field, "target:", diagnostic["target"])
        print("  four levels:", diagnostic["sizes"])
        print("  three successive orders:", diagnostic["orders"])
        print("  fitted order:", diagnostic["fitted_order"])
        if "amplitude_ratio" in diagnostic:
            print("  max/min target-normalized amplitude:", diagnostic["amplitude_ratio"])


## References and scope

Each method's original references, approximation hypotheses and independent numerical controls appear on its [tutorial](https://ipes-lncc.github.io/pymhm/tutorials/methods/) and in the [Bibliography](https://ipes-lncc.github.io/pymhm/literature/). The raw records retain their execution identities. Smooth estimates do not apply automatically to SPE10 or a coefficient interface. Unfitted trace refinement holds the macro mesh fixed; temporal refinement holds the spatial discretization fixed.

The recovery tutorials check recovered flux and projected-divergence convergence, physical conservation and estimator effectivity. Adaptive studies compare error against work; recursive studies compare equivalent leaf discretizations and physical errors. Their indicators and conservation invariants are reported alongside applicable convergence estimates.
